# Stage 7 — Multimodal Clinical + Fundus Feature Fusion
**Capstone: Multimodal Ocular Disease Prediction**

Runs entirely on your local machine (RTX 4060, VS Code, `venv311`) — same setup as Stage 6.

Task: **Myopia vs Non-Myopia, at patient level**, combining:
- a small clinical branch (Patient Age, Patient Sex — the same 2 legitimate features used in Stage 3/4, still avoiding diagnostic-keyword / disease-indicator leakage)
- an image branch built from the **already-trained Stage 6 MobileNetV2 checkpoint**, used here as a frozen feature extractor
- a small fusion classifier trained on the combined features

**Data source change from Stage 6:** Stage 6 used `datasets/myopia` and `datasets/normal` — real, useful images, but with **no patient ID in the filenames**, so they could not be linked to `full_df.csv`. This stage instead uses `full_df.csv` joined to `preprocessed_images/`, where every filename (`{ID}_left.jpg` / `{ID}_right.jpg`) is directly traceable to a real patient ID and real clinical fields. `Code/verify_multimodal_linkage.py` was run first and confirmed **6,392/6,392 referenced images exist on disk** and **0 patients have conflicting left/right Myopia labels** — this pairing is genuine, not assumed.

**Before running this notebook:** `Code/gpu_check.py` should already report the RTX 4060 as detected (confirmed in Stage 6). This notebook reuses that same environment.

## 1. Confirm the GPU

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("=" * 70)
    print("WARNING: no GPU detected -- fix this before trusting any results here.")
    print("=" * 70)


Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Load full_df.csv and build a patient-level cohort

`full_df.csv` is image-level (one row per eye), not patient-level. For each patient we take:
- **Patient Age**, **Patient Sex** (identical across a patient's rows — these are the only 2 legitimate clinical features, same exclusions as Stage 2/4: no diagnostic keywords, no other disease indicator columns)
- **Myopia flag** — 1 if any of the patient's rows has `M == 1`, else 0
- the list of that patient's available image filenames (1 or 2, left/right)

Every filename is checked against `preprocessed_images/` again here (defensive re-check, not just trusting the earlier verification script) — any patient whose image is unexpectedly missing is dropped rather than guessed.

In [2]:
import os
import pandas as pd
import numpy as np

CANDIDATE_ROOTS = ["../..", ".", "../../.."]
DATA_ROOT = next((r for r in CANDIDATE_ROOTS if os.path.isfile(os.path.join(r, "full_df.csv"))), None)
assert DATA_ROOT is not None, "Could not locate full_df.csv -- run this notebook from Code/notebooks/"
DATA_ROOT = os.path.abspath(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT)

CSV_PATH = os.path.join(DATA_ROOT, "full_df.csv")
IMG_DIR = os.path.join(DATA_ROOT, "preprocessed_images")

df = pd.read_csv(CSV_PATH)
print("full_df.csv shape:", df.shape)

# Patient-level aggregation
patient_rows = []
for pid, g in df.groupby("ID"):
    age = g["Patient Age"].iloc[0]
    sex = g["Patient Sex"].iloc[0]
    myopia_flag = int((g["M"] == 1).any())
    filenames = [f for f in g["filename"].tolist() if os.path.isfile(os.path.join(IMG_DIR, f))]
    if len(filenames) == 0:
        continue  # no usable image for this patient -- excluded, not guessed
    patient_rows.append({"ID": pid, "Patient Age": age, "Patient Sex": sex,
                          "Myopia": myopia_flag, "filenames": filenames})

cohort = pd.DataFrame(patient_rows)
print("Patients in cohort (>=1 real linked image):", len(cohort))
print("Myopia-positive patients:", cohort["Myopia"].sum(),
      f"({100*cohort['Myopia'].mean():.2f}%)")
print("Patients with 2 linked images:", (cohort["filenames"].apply(len) == 2).sum())
print("Patients with 1 linked image:", (cohort["filenames"].apply(len) == 1).sum())
cohort.to_csv(os.path.join(DATA_ROOT, "Data", "stage7_multimodal_cohort.csv"), index=False)
print("Saved Data/stage7_multimodal_cohort.csv")


DATA_ROOT: c:\Users\PRAJWAL\OneDrive\Desktop\Capstone Datasets
full_df.csv shape: (6392, 19)
Patients in cohort (>=1 real linked image): 3358
Myopia-positive patients: 171 (5.09%)
Patients with 2 linked images: 3034
Patients with 1 linked image: 324
Saved Data/stage7_multimodal_cohort.csv


## 3. Patient-level train / validation / test split (stratified, grouped by construction)

Because the cohort table above already has exactly one row per patient, a normal stratified `train_test_split` on this table is automatically patient-grouped -- no patient's images can appear in two splits.

In [3]:
from sklearn.model_selection import train_test_split

train_cohort, temp_cohort = train_test_split(
    cohort, test_size=0.30, random_state=42, stratify=cohort["Myopia"]
)
val_cohort, test_cohort = train_test_split(
    temp_cohort, test_size=0.50, random_state=42, stratify=temp_cohort["Myopia"]
)

for name, part in [("Train", train_cohort), ("Val", val_cohort), ("Test", test_cohort)]:
    print(f"{name}: {len(part)} patients | Myopia rate: {part['Myopia'].mean():.4f}")


Train: 2350 patients | Myopia rate: 0.0511
Val: 504 patients | Myopia rate: 0.0516
Test: 504 patients | Myopia rate: 0.0496


## 4. Clinical branch — features and scaling (fit on train only)

Same 2 legitimate features as Stage 3/4: Patient Age (median-imputed, standardized) and Patient Sex (encoded). Imputer and scaler are fit on the training patients only, then applied to val/test -- no leakage from val/test statistics.

In [4]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def encode_sex(s):
    return 1 if str(s).strip().lower().startswith("m") else 0

age_imputer = SimpleImputer(strategy="median")
age_imputer.fit(train_cohort[["Patient Age"]])

age_scaler = StandardScaler()
age_scaler.fit(age_imputer.transform(train_cohort[["Patient Age"]]))

def clinical_features(part_df):
    age_imputed = age_imputer.transform(part_df[["Patient Age"]])
    age_scaled = age_scaler.transform(age_imputed).flatten()
    sex_encoded = part_df["Patient Sex"].apply(encode_sex).values
    return np.stack([age_scaled, sex_encoded], axis=1).astype("float32")

train_clin = clinical_features(train_cohort)
val_clin = clinical_features(val_cohort)
test_clin = clinical_features(test_cohort)
print("Clinical feature shape (train):", train_clin.shape)
print("Example rows:", train_clin[:3])


Clinical feature shape (train): (2350, 2)
Example rows: [[-1.0905656   1.        ]
 [ 1.1970695   1.        ]
 [-0.41274777  1.        ]]


## 5. Image branch — load the trained Stage 6 MobileNetV2 checkpoint as a feature extractor

Same architecture as `06_cnn_local_gpu_training.ipynb`: MobileNetV2 backbone (frozen, ImageNet weights) + a small trained head (`Dropout -> Linear(1280,64) -> ReLU -> Dropout -> Linear(64,1)`). The Stage 6 checkpoint is loaded in full, then the **64-dimensional layer right before the final classification layer** is used as the image feature vector -- this genuinely reuses the Stage 6 trained weights, not just generic ImageNet features.

Note: the Stage 6 checkpoint was trained on `datasets/myopia` + `datasets/normal` images; here it is applied to `preprocessed_images/` (a different source folder, same kind of fundus photo). No further training happens to the backbone or the reused head here -- it is used purely as a fixed feature extractor, consistent with a feature-fusion design.

In [5]:
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights
import torch.nn as nn
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

base_model = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
num_features = base_model.classifier[1].in_features
base_model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 1),
)

CKPT_PATH = os.path.join(DATA_ROOT, "Code", "models", "stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt")
state_dict = torch.load(CKPT_PATH, map_location=device)
base_model.load_state_dict(state_dict)
base_model = base_model.to(device)
base_model.eval()
for p in base_model.parameters():
    p.requires_grad = False

# Truncate classifier: keep Dropout -> Linear(1280,64) -> ReLU (stop before the final Dropout+Linear(64,1))
feature_extractor = base_model
feature_head = nn.Sequential(*list(base_model.classifier.children())[:3]).to(device)  # Dropout, Linear, ReLU
print("Loaded Stage 6 checkpoint from:", CKPT_PATH)
print("Feature head (first 3 classifier layers):", feature_head)


Loaded Stage 6 checkpoint from: c:\Users\PRAJWAL\OneDrive\Desktop\Capstone Datasets\Code\models\stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt
Feature head (first 3 classifier layers): Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1280, out_features=64, bias=True)
  (2): ReLU()
)


## 6. Extract a fixed 64-d image feature vector per patient

For patients with 2 linked images (left + right), the two 64-d feature vectors are averaged into one. This is a single forward pass per image (no augmentation, no gradient) -- computed once for every patient in the cohort so it does not need to be repeated per training epoch.

In [6]:
from torch.utils.data import Dataset, DataLoader

class PatientImageDataset(Dataset):
    """Yields every (patient_index, image_tensor) pair, one row per image."""
    def __init__(self, cohort_df):
        self.items = []
        for i, row in cohort_df.reset_index(drop=True).iterrows():
            for fname in row["filenames"]:
                self.items.append((i, fname))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        i, fname = self.items[idx]
        img = Image.open(os.path.join(IMG_DIR, fname)).convert("RGB")
        img = eval_transform(img)
        return i, img

def extract_patient_image_features(cohort_df, batch_size=32):
    cohort_df = cohort_df.reset_index(drop=True)
    n = len(cohort_df)
    sums = np.zeros((n, 64), dtype="float32")
    counts = np.zeros(n, dtype="int32")

    ds = PatientImageDataset(cohort_df)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    with torch.no_grad():
        for idxs, imgs in loader:
            imgs = imgs.to(device)
            feats = feature_extractor.features(imgs)
            feats = nn.functional.adaptive_avg_pool2d(feats, 1).flatten(1)
            feats = feature_head(feats)  # -> (batch, 64)
            feats = feats.cpu().numpy()
            for j, i in enumerate(idxs.numpy()):
                sums[i] += feats[j]
                counts[i] += 1

    counts = np.maximum(counts, 1)
    return sums / counts[:, None]

print("Extracting image features for train/val/test patients (one forward pass per image)...")
train_img = extract_patient_image_features(train_cohort)
val_img = extract_patient_image_features(val_cohort)
test_img = extract_patient_image_features(test_cohort)
print("Image feature shapes:", train_img.shape, val_img.shape, test_img.shape)


Extracting image features for train/val/test patients (one forward pass per image)...
Image feature shapes: (2350, 64) (504, 64) (504, 64)


## 7. Fusion model — clinical branch + image branch -> combined classifier

- Clinical branch: `Linear(2, 8) -> ReLU` (turns [age, sex] into 8 learned clinical features)
- Image branch: the 64-d vector extracted above (already fixed/frozen)
- Fusion: concatenate `[image(64), clinical(8)]` -> `Linear(72,32) -> ReLU -> Dropout(0.3) -> Linear(32,1)`

Only the fusion model (clinical branch + combiner) is trained here -- the image feature extractor stays frozen exactly as loaded from Stage 6.

In [7]:
class FusionModel(nn.Module):
    def __init__(self, clinical_dim=2, image_dim=64, clinical_hidden=8, fusion_hidden=32):
        super().__init__()
        self.clinical_branch = nn.Sequential(
            nn.Linear(clinical_dim, clinical_hidden),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(image_dim + clinical_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(fusion_hidden, 1),
        )

    def forward(self, image_feat, clinical_feat):
        clin = self.clinical_branch(clinical_feat)
        combined = torch.cat([image_feat, clin], dim=1)
        return self.classifier(combined)

fusion_model = FusionModel().to(device)
trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
print("Fusion model trainable parameters:", trainable)
print(fusion_model)


Fusion model trainable parameters: 2393
FusionModel(
  (clinical_branch): Sequential(
    (0): Linear(in_features=2, out_features=8, bias=True)
    (1): ReLU()
  )
  (classifier): Sequential(
    (0): Linear(in_features=72, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)


## 8. Train the fusion model

Class-weighted `BCEWithLogitsLoss` (Myopia is a small minority at patient level too), early stopping on validation loss (patience 5, max 40 epochs -- this trains fast since only ~377 trainable parameters and precomputed features are used, no image loading during training).

In [8]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(train_cohort["Myopia"])
cw = compute_class_weight("balanced", classes=classes, y=train_cohort["Myopia"])
pos_weight = torch.tensor(cw[1] / cw[0], dtype=torch.float32).to(device)
print("Class weights (Non-Myopia, Myopia):", cw, "-> pos_weight:", pos_weight.item())

train_img_t = torch.tensor(train_img, dtype=torch.float32).to(device)
train_clin_t = torch.tensor(train_clin, dtype=torch.float32).to(device)
train_y_t = torch.tensor(train_cohort["Myopia"].values, dtype=torch.float32).to(device)

val_img_t = torch.tensor(val_img, dtype=torch.float32).to(device)
val_clin_t = torch.tensor(val_clin, dtype=torch.float32).to(device)
val_y_t = torch.tensor(val_cohort["Myopia"].values, dtype=torch.float32).to(device)

optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

EPOCHS = 40
PATIENCE = 5
best_val_loss = float("inf")
epochs_no_improve = 0
best_state = None
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(EPOCHS):
    fusion_model.train()
    optimizer.zero_grad()
    logits = fusion_model(train_img_t, train_clin_t).squeeze(1)
    loss = criterion(logits, train_y_t)
    loss.backward()
    optimizer.step()
    train_acc = ((torch.sigmoid(logits) > 0.5).float() == train_y_t).float().mean().item()

    fusion_model.eval()
    with torch.no_grad():
        val_logits = fusion_model(val_img_t, val_clin_t).squeeze(1)
        val_loss = criterion(val_logits, val_y_t).item()
        val_acc = ((torch.sigmoid(val_logits) > 0.5).float() == val_y_t).float().mean().item()

    history["train_loss"].append(loss.item())
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} - train_loss: {loss.item():.4f} train_acc: {train_acc:.4f} "
          f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_state = {k: v.clone() for k, v in fusion_model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no val_loss improvement for {PATIENCE} epochs).")
            break

if best_state is not None:
    fusion_model.load_state_dict(best_state)
    print("Restored best-val-loss weights.")


Class weights (Non-Myopia, Myopia): [0.52690583 9.79166667] -> pos_weight: 18.58333396911621
Epoch 1/40 - train_loss: 1.3788 train_acc: 0.2396 val_loss: 1.2960 val_acc: 0.0536
Epoch 2/40 - train_loss: 1.3046 train_acc: 0.2915 val_loss: 1.2325 val_acc: 0.2817
Epoch 3/40 - train_loss: 1.2328 train_acc: 0.3991 val_loss: 1.1733 val_acc: 0.5813
Epoch 4/40 - train_loss: 1.1984 train_acc: 0.4740 val_loss: 1.1176 val_acc: 0.7460
Epoch 5/40 - train_loss: 1.1267 train_acc: 0.5694 val_loss: 1.0653 val_acc: 0.8274
Epoch 6/40 - train_loss: 1.0721 train_acc: 0.6549 val_loss: 1.0166 val_acc: 0.8591
Epoch 7/40 - train_loss: 1.0348 train_acc: 0.7243 val_loss: 0.9717 val_acc: 0.8790
Epoch 8/40 - train_loss: 1.0120 train_acc: 0.7753 val_loss: 0.9305 val_acc: 0.8909
Epoch 9/40 - train_loss: 0.9369 train_acc: 0.8187 val_loss: 0.8928 val_acc: 0.9008
Epoch 10/40 - train_loss: 0.9244 train_acc: 0.8345 val_loss: 0.8584 val_acc: 0.9048
Epoch 11/40 - train_loss: 0.8719 train_acc: 0.8634 val_loss: 0.8269 val_acc:

## 9. Training curves

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["train_loss"], label="train loss")
axes[0].plot(history["val_loss"], label="val loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train accuracy")
axes[1].plot(history["val_acc"], label="val accuracy")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig("stage7_fusion_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved stage7_fusion_training_curves.png")


Saved stage7_fusion_training_curves.png


C:\Users\PRAJWAL\AppData\Local\Temp\ipykernel_18808\1149528943.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Evaluate on the held-out test patients

In [10]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

test_img_t = torch.tensor(test_img, dtype=torch.float32).to(device)
test_clin_t = torch.tensor(test_clin, dtype=torch.float32).to(device)
test_y = test_cohort["Myopia"].values

fusion_model.eval()
with torch.no_grad():
    test_logits = fusion_model(test_img_t, test_clin_t).squeeze(1)
    test_probs = torch.sigmoid(test_logits).cpu().numpy()
test_preds = (test_probs > 0.5).astype(int)

acc = accuracy_score(test_y, test_preds)
prec = precision_score(test_y, test_preds, pos_label=1, zero_division=0)
rec = recall_score(test_y, test_preds, pos_label=1, zero_division=0)
f1 = f1_score(test_y, test_preds, pos_label=1, zero_division=0)
f1_macro = f1_score(test_y, test_preds, average="macro", zero_division=0)

print("Device used for this run:", device)
print(f"Test accuracy:  {acc:.4f}")
print(f"Precision (Myopia): {prec:.4f}")
print(f"Recall (Myopia):    {rec:.4f}")
print(f"F1 (Myopia):        {f1:.4f}")
print(f"F1 (macro):         {f1_macro:.4f}")
print()
print(classification_report(test_y, test_preds, target_names=["Non-Myopia", "Myopia"]))

cm = confusion_matrix(test_y, test_preds)
print("Confusion matrix (rows=actual, cols=predicted):")
print(cm)


Device used for this run: cuda
Test accuracy:  0.9048
Precision (Myopia): 0.3380
Recall (Myopia):    0.9600
F1 (Myopia):        0.5000
F1 (macro):         0.7237

              precision    recall  f1-score   support

  Non-Myopia       1.00      0.90      0.95       479
      Myopia       0.34      0.96      0.50        25

    accuracy                           0.90       504
   macro avg       0.67      0.93      0.72       504
weighted avg       0.96      0.90      0.93       504

Confusion matrix (rows=actual, cols=predicted):
[[432  47]
 [  1  24]]


## 11. Confusion matrix plot

In [11]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Non-Myopia", "Myopia"], yticklabels=["Non-Myopia", "Myopia"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Stage 7 fusion -- test set confusion matrix (device={device})")
plt.tight_layout()
plt.savefig("stage7_fusion_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved stage7_fusion_confusion_matrix.png")


Saved stage7_fusion_confusion_matrix.png


C:\Users\PRAJWAL\AppData\Local\Temp\ipykernel_18808\633107342.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Compare against the Stage 4 clinical baseline and Stage 6 image baseline

Important caveat: these three are **not evaluated on identical splits or identical tasks**, so treat this as a directional comparison, not a strict apples-to-apples benchmark:
- Stage 4 (clinical only): image-level rows (not patient-grouped for this specific table), 2 legitimate features only.
- Stage 6 (image only): image-level, `datasets/myopia` + `datasets/normal` (no patient ID available on that source).
- Stage 7 (fusion, this notebook): **patient-level**, grouped train/val/test split, `full_df.csv` + `preprocessed_images/`.

In [12]:
comparison = pd.DataFrame([
    {"Model": "Stage 4 -- Random Forest (clinical only)", "Accuracy": 0.7477, "Precision (Myopia)": 0.0426,
     "Recall (Myopia)": 0.2889, "F1 (Myopia)": 0.0743, "F1 (macro)": 0.4641},
    {"Model": "Stage 4 -- Logistic Regression (clinical only)", "Accuracy": 0.5950, "Precision (Myopia)": 0.0527,
     "Recall (Myopia)": 0.6222, "F1 (Myopia)": 0.0972, "F1 (macro)": 0.4181},
    {"Model": "Stage 6 -- MobileNetV2 (image only, local GPU)", "Accuracy": 0.9657, "Precision (Myopia)": 0.6939,
     "Recall (Myopia)": 0.9714, "F1 (Myopia)": 0.8095, "F1 (macro)": 0.8953},
    {"Model": "Stage 7 -- Multimodal fusion (this run)", "Accuracy": acc, "Precision (Myopia)": prec,
     "Recall (Myopia)": rec, "F1 (Myopia)": f1, "F1 (macro)": f1_macro},
])
comparison.to_csv(os.path.join(DATA_ROOT, "Data", "stage7_model_comparison.csv"), index=False)
print(comparison.to_string(index=False))
print()
print("Saved Data/stage7_model_comparison.csv")


                                         Model  Accuracy  Precision (Myopia)  Recall (Myopia)  F1 (Myopia)  F1 (macro)
      Stage 4 -- Random Forest (clinical only)  0.747700            0.042600           0.2889       0.0743    0.464100
Stage 4 -- Logistic Regression (clinical only)  0.595000            0.052700           0.6222       0.0972    0.418100
Stage 6 -- MobileNetV2 (image only, local GPU)  0.965700            0.693900           0.9714       0.8095    0.895300
       Stage 7 -- Multimodal fusion (this run)  0.904762            0.338028           0.9600       0.5000    0.723684

Saved Data/stage7_model_comparison.csv


## 13. Save the fusion model checkpoint

In [13]:
MODELS_DIR = os.path.join(DATA_ROOT, "Code", "models")
os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, "stage7_multimodal_fusion_local_gpu.pt")
torch.save(fusion_model.state_dict(), model_path)
print("Saved fusion model state_dict to:", model_path)


Saved fusion model state_dict to: c:\Users\PRAJWAL\OneDrive\Desktop\Capstone Datasets\Code\models\stage7_multimodal_fusion_local_gpu.pt


## 14. Summary

| Item | Value |
|---|---|
| Task | Binary: Myopia vs Non-Myopia, **patient level** |
| Data source | `full_df.csv` joined to `preprocessed_images/` via `{ID}_left/right.jpg` -- verified 6,392/6,392 images present, 0 label conflicts |
| Cohort | Patients with >=1 linked image; split 70/15/15, stratified, grouped by construction (one row per patient) |
| Clinical branch | Patient Age (imputed+scaled) + Patient Sex -> Linear(2,8)+ReLU |
| Image branch | Stage 6's trained MobileNetV2 checkpoint, frozen, truncated to its 64-d pre-classifier layer |
| Fusion | concat(64+8) -> Linear(72,32)+ReLU+Dropout(0.3) -> Linear(32,1) |
| Framework / device | PyTorch -- printed `device` above records whether cuda or cpu actually ran |
